# Batch Inference for Fine-tuned Models (Vertex AI Custom Job)

This notebook implements batch inference for fine-tuned models using Cloud Vertex AI. We use batch inference because:
1. We want to run inference on the test dataset that the model has not seen during training for evaluation.
2. Real-time inference would be too costly and slow for our dataset size
3. We use Vertex AI aiplatform.CustomJob to have the option to run inference in the local environment or with GPU instances remotely without code changes.

**FLow**
```
GCS (test data + model adapter)
        ↓
Vertex AI Custom Job (GPU)
  └─ infer.py: download → vLLM infer → upload results
        ↓
GCS (results.jsonl)
        ↓
Notebook: download → tracking CSV → Notebook 05 evaluate
```

In [1]:
print("Notebook 04 run")

Notebook 04 run


In [2]:
import sys
from utils.training_image import (
    # get_sagemaker_distribution, 
    # SageMakerDistribution, 
    get_python_version, 
    # get_aws_account_id_for_region,
    is_docker_installed,
    is_docker_compose_installed,
    # check_and_enable_docker_access_sagemaker_studio
)

In [3]:
py_version = sys.version_info
python_version = str(get_python_version(*py_version))
print(f"Your Python version: {python_version}")

Your Python version: 3.12


In [4]:
# sm_distro_version = get_sagemaker_distribution(py_version)
# print(f"Using SageMaker distribution v{sm_distro_version.image_version} as training image.")

## Import Required Libraries

In [5]:
# sagemaker_sdk_version = sm_distro_version.sagemaker_python_sdk # local SageMaker version must be same as in training job with remote decorator

In [ ]:
%pip install -U --quiet requests beautifulsoup4 dataclasses
# %pip install --quiet sagemaker=={sagemaker_sdk_version} 

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'sagemaker=={sagemaker_sdk_version}': Expected semicolon (after name with no version specifier) or end
    sagemaker=={sagemaker_sdk_version}
             ^


In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
import json, os, subprocess, csv
from pathlib import Path
import pandas as pd
from typing import Union, Dict, Optional
from IPython.display import display, HTML
from ipywidgets import widgets

from google.cloud import aiplatform, storage

In [9]:
import os, json
print(os.getcwd())
print(os.path.exists("gcs_config.json"))

d:\Internship-Biwoco\Fine-tune\sample-for-multi-modal-document-to-json-with-vertex-ai
True


In [10]:
# Initialize session and configure GCP resources for training
try:
    with open("gcs_config.json") as f:
        gcs_cfg = json.load(f)

    PROJECT_ID          = gcs_cfg["project_id"]
    default_bucket_name = gcs_cfg["bucket_name"]
    region              = gcs_cfg["region"]
    dataset_gcs_prefix  = gcs_cfg["gcs_output_prefix"]
    gcs_root_uri        = f"gs://{default_bucket_name}"
    dataset_gcs_uri     = f"{gcs_root_uri}/{dataset_gcs_prefix}"

    gcs_model_dir = gcs_cfg.get("gcs_model_dir", f"{gcs_root_uri}/output")
    gcs_results   = gcs_cfg.get("gcs_results_dir", f"{gcs_root_uri}/inference_results")

    aiplatform.init(
        project=PROJECT_ID,
        location=region,
        staging_bucket=gcs_root_uri,  
    )


except Exception as e:
    raise Exception(f"Error setting up GCP session: {str(e)}")

print(" Initialized Vertex AI session...")
print(f" Using dataset : {dataset_gcs_uri}")
print(f" Model dir     : {gcs_model_dir}")
print(f" Results dir   : {gcs_results}")

 Initialized Vertex AI session...
 Using dataset : gs://electric-bill-dataset-gcs/data/swift_dataset
 Model dir     : gs://electric-bill-dataset-gcs/output
 Results dir   : gs://electric-bill-dataset-gcs/inference_results


## Retrieve model artifact location from fine-tuning

In [11]:
from utils.config import ModelConfig

# The models that you fine-tuned.
base_model_config = ModelConfig(
    # Replace with model type and model id of the base model.
    model_type="qwen2_5_vl",
    model_id="Qwen/Qwen2.5-VL-3B-Instruct"

    # model_type = "llama3_2_vision",
    # model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
)

print("✅ Configured model id.")

✅ Configured model id.


In [12]:
training_job_name_prefix = base_model_config.training_job_prefix(dataset_gcs_prefix)
print(f"Fine-tuning name prefix: {training_job_name_prefix}")


Fine-tuning name prefix: finetune-qwen2-5-vl-3b-instruct-data-sw


In [13]:
display(HTML(f"""
<div style="border: 2px solid #006CE0; 
    padding: 10px; 
    border-radius: 5px; 
    max-width: 100%;
    background: #f0fbff;">
    <b>Note:</b> Skip the next 4 cells below if you want to run inference using <b>{base_model_config.model_id}</b> from HuggingFace Hub. 
    <br>Run the cells below if you want to use a model that you have fine-tuned.
</div>
"""))

In [14]:
from utils.model_manager import list_gcs_models, list_available_models 

df_models = list_gcs_models(gcs_model_dir)
print(f"Tìm thấy {len(df_models)} objects trong {gcs_model_dir}:")
display(df_models)

Tìm thấy 1 objects trong gs://electric-bill-dataset-gcs/output:


,Key
0,gs://electric-bill-dataset-gcs/output/model/


In [15]:
which_model_to_pick = 0 # use first model from list by default. Change to use a different model from list above.

In [16]:
# Set up the URI from which we will download the model
model_output_url = df_models['Key'].iloc[which_model_to_pick]
print(f"Selected model: {model_output_url}")

Selected model: gs://electric-bill-dataset-gcs/output/model/


In [17]:
# model_suffix_s3 = get_s3_suffix(model_output_url)

<div style="border: 2px solid #006CE0; 
    padding: 10px; 
    border-radius: 5px; 
    max-width: 100%;
    background: #f0fbff;">
    Continue below for inference with base model or fine-tuned model.
</div>

In [18]:
try:
    model_config = ModelConfig(
        # Replace with model type and model id of the base model.
        model_type=base_model_config.model_type,
        model_id=model_output_url
    )
    prefix = model_config.model_id.replace("/","-").replace(".","-")
    
    # prefix = model_suffix_s3.split("/")[0]
    # print("✅ Configured fine-tuned model id.")
    
except NameError:
    # not using fine-tuned model
    model_config = base_model_config
    prefix = model_config.model_id.replace("/","-").replace(".","-")
    print("✅ Using base model for inference.")

In [19]:
print(f"Model for inference: {model_config.model_id}")

Model for inference: gs://electric-bill-dataset-gcs/output/model/


## Configure Job for Batch Inference

Lets define the Vertex AI Custom Job configuration

In [20]:
# sagemaker_distr_account_id = get_aws_account_id_for_region(region)
# if not sagemaker_distr_account_id:
#     raise ValueError(
#         f"Please make sure to manually set the `sagemaker_distr_account_id` account id for your specific AWS region ({region}) from the AWS documentation: https://docs.aws.amazon.com/sagemaker/latest/dg/notebooks-available-images.html#notebooks-available-images-arn"
#     )

In [21]:
# # lets define the sagemaker distribution to use
# sagemaker_dist_uri = f"{sagemaker_distr_account_id}.dkr.ecr.{region}.amazonaws.com/sagemaker-distribution-prod:{sm_distro_version.image_version}-gpu"

In [22]:
# = config matching notebook 03
CONTAINER_URI     = "asia-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-3.py310:latest"
MACHINE_TYPE      = "a2-highgpu-1g"
ACCELERATOR_TYPE  = "NVIDIA_TESLA_A100"
ACCELERATOR_COUNT = 1

Define the dependencies that are required for inference.

In [23]:
# matching notebook 03
requirements = [
    "ms-swift==3.2.2",
    "transformers==4.49.0",
    "qwen_vl_utils==0.0.11",
    "accelerate==1.1.0",
    "tensorboard",
    "tensorboardX",
    "decord",
    "av",
    "hf_transfer",
]

In [24]:
# %store requirements >requirements.txt

In [25]:
# s3_root_uri = "s3://{}".format(default_bucket_name)

### Environment Variables Configuration

We set specific environment variables because:
1. Memory usage needs to be optimized for GPUs
2. Image processing has size constraints
3. We want faster downloads from Hugging Face
4. Resource limits need to be carefully managed

In [26]:
# defines the environment variables for the training
env_variables ={
    "SIZE_FACTOR": json.dumps(8), # can be increase but requires more GPU memory
    "MAX_PIXELS": json.dumps(1048576), # can be increase but requires more GPU memory
    "USE_HF_TRANSFER": json.dumps(1),
    "HF_HUB_ENABLE_HF_TRANSFER": json.dumps(1),
    # "HF_TOKEN": "xxxxxxxx",
}


In [27]:
from datetime import datetime
timestamp       = datetime.now().strftime("%Y%m%d%H%M%S")
job_name_prefix = f"infer-{prefix}-{timestamp}"[:60]
print(f"Job name: {job_name_prefix}")

Job name: infer-gs:--electric-bill-dataset-gcs-output-model--202606241


### Constrained Decoding

Constrained decoding controls a language model's next-token prediction process by limiting which tokens it can generate to only those that satisfy specific rules or formats. During the normal generation process, a language model assigns probabilities to all possible next tokens. With constrained decoding the set of next tokens is limited to only tokens that satisfy the required structure. For example with JSON constrained decoding the model can only select tokens that create a valid JSON syntax. 

Below you can configure constrained decoding for the batch inference:
1. Set it to `None` to run batch inference without any constrained decoding.
2. If you have a JSON schema file in your dataset you can set `guided_decoding` to the path of that JSON schema file inside your dataset, for example `guided_decoding = "groundtruth_schema.json"`. You can reference the [02_create_custom_dataset_swift.ipynb](02_create_custom_dataset_swift.ipynb) notebook on how to create a JSON schema file. 
3. You can also set `guided_decoding` to a dict sturctured output parameter from the [vLLM documentation](https://docs.vllm.ai/en/latest/features/structured_outputs.html), for example `guided_decoding = {"guided_choice": ["positive", "negative"]}`

In [28]:
guided_decoding = None # 1. default no constrained decoding

# guided_decoding = "groundtruth_schema.json" # 2. use a JSON schema inside dataset

# 3. Below is an example on how to configure structure output in accordance to the vLLM documentation
# from pydantic import BaseModel

# class Invoice(BaseModel):
#     purpose: str
#     amount: int

# json_schema = Invoice.model_json_schema()
# guided_decoding = {"guided_json": json_schema}

## Batch Inference Function

In [29]:
# @remote(
#     instance_type="ml.g6e.xlarge",  # Powerful GPU for fast inference
#     instance_count=1,  # Single instance for cost efficiency
#     volume_size=200, # Large volume for model and data storage
#     job_name_prefix=job_name_prefix,
#     # use_spot_instances=True, # Cost efficient inference. Inference can be restarted if no spot capacity. 
#     max_wait_time_in_seconds=172800, # 48 hours max wait
#     max_runtime_in_seconds=172800, # 48 hours max runtime
# )
# def batch_inference(
#     model_id: str,
#     model_type: str,
#     dataset_s3: str,
#     test_data_path: str = "test.jsonl",
#     guided_decoding: Optional[Union[Dict, str]] = None
# ) -> str:
#     """
#     Run batch inference using SageMaker.
    
#     Args:
#         model_id: Model identifier or S3 URI
#         model_type: Type of the model
#         dataset_s3: S3 URI for the dataset
#         test_data_path: Path to the test data file
#         guided_decoding: vllm guided_decoding config or path to json schema. Default: None - no constrained decoding used
        
#     Returns:
#         Status message
#     """
#     from utils.model_manager import ModelManager
#     from swift.llm import infer_main
#     from pathlib import Path
#     import subprocess
#     import json


#     output_dir = Path("/opt/ml/model")
    
#     # copy the training data from input source to local directory
#     dataset_dir = Path(".")
#     os.makedirs(dataset_dir, exist_ok=True)
#     subprocess.run(
#         ["aws", "s3", "cp", dataset_s3, dataset_dir, "--recursive", "--quiet"],
#         shell = False
#     )
    
#     test_data_local_path = dataset_dir / test_data_path
#     result_path = output_dir / "results.jsonl"
    
#     model_manager = ModelManager()
#     guided_decoding = model_manager.construct_guided_decoding_config(dataset_dir, guided_decoding)
    
#     argv = [
#         "--result_path", str(result_path),
#         "--max_length", "4096",  # Maximum sequence length for processing
#         "--load_data_args", "false",
#         "--val_dataset", str(test_data_local_path),
#         "--use_hf", "true", 
#         "--infer_backend", "vllm",  # Use VLLM for faster inference
#         "--gpu_memory_utilization", "0.95",  # High GPU utilization for speed
#         "--max_num_seqs", "8",  # Batch size for parallel processing
#         "--limit_mm_per_prompt", '{"image": 1, "video": 0}', # how many images per prompt. Increase if you have multi page pdf
#         "--temperature", "0",
#     ]

#     model_dir: Path
        
#     # Handle model loading
#     if model_id.startswith("s3://"):
        
#         model_dir = model_manager.download_and_extract_model(model_id)
#         ckpt_dir = model_manager.find_best_model_checkpoint(model_dir)
       
        
#         model_ckpt_args = [
#             "--adapters", str(ckpt_dir),
#             "--merge_lora", "true"
#         ]
#         argv.extend(model_ckpt_args)
        
#     else:
#         model_dir = model_manager.download_from_hf_hub(model_id)
#         from_hub_args = ["--model_type", model_type, "--model", str(model_dir)]
#         argv.extend(from_hub_args)

#     model_manager.update_generation_config(model_dir, guided_decoding)

#     result = infer_main(argv)
#     return "done"

In [30]:
def batch_inference(model_id, model_type, dataset_gcs,
                    test_data_path="conversations_test_swift_format.json",
                    guided_decoding=None):

    run_results_gcs = f"{gcs_results}/{job_name_prefix}"

    job = aiplatform.CustomJob.from_local_script(
        display_name=job_name_prefix,
        script_path="infer.py",           # = train.py in notebook 03
        container_uri=CONTAINER_URI,      # = notebook 03
        requirements=requirements,        # = notebook 03 + thêm vllm
        machine_type=MACHINE_TYPE,
        accelerator_type=ACCELERATOR_TYPE,
        accelerator_count=ACCELERATOR_COUNT,
        base_output_dir=run_results_gcs,
        args=[
            "--model_id",        model_id,
            "--model_type",      model_type,
            "--dataset_gcs",     dataset_gcs,
            "--test_data_path",  test_data_path,
            "--results_gcs",     run_results_gcs,
            "--guided_decoding", json.dumps(guided_decoding),
        ],
        environment_variables=env_variables,
    )

    print(f" Submitting job: {job_name_prefix}")
    print(f" View: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={PROJECT_ID}")
    job.run(sync=True)
    return run_results_gcs

## Run Batch Inference

In [31]:
inference_kwargs = {
    "model_id":        model_config.model_id,
    "model_type":      model_config.model_type,
    "dataset_gcs":      dataset_gcs_uri,   
    "test_data_path":  "conversations_test_swift_format.json",
    "guided_decoding": guided_decoding
}

In [ ]:
print(
    f"View your job here: https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/jobs/"
)
batch_inference(**inference_kwargs)

View your job here: https://asia-southeast1.console.aws.amazon.com/sagemaker/home?region=asia-southeast1#/jobs/
Training script copied to:
gs://electric-bill-dataset-gcs/aiplatform-2026-06-24-14:18:47.002-aiplatform_custom_trainer_script-0.1.tar.gz.
 Submitting job: infer-gs:--electric-bill-dataset-gcs-output-model--202606241
 View: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project=first-orc-chien
Creating CustomJob
CustomJob created. Resource name: projects/73397202200/locations/asia-southeast1/customJobs/8170502516962754560
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/73397202200/locations/asia-southeast1/customJobs/8170502516962754560')
View Custom Job:
https://console.cloud.google.com/agent-platform/locations/asia-southeast1/training/8170502516962754560?project=73397202200
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/8170502516962754560 current state:
JOB_STATE_PENDING
CustomJob projects/73397

In [ ]:
inference_output_url = f"{gcs_results}/{job_name_prefix}"
print(f"📊 Inference results: {inference_output_url}/results.jsonl")

📊 Inference results: gs://electric-bill-dataset-gcs/inference_results/infer-gs:--electric-bill-dataset-gcs-output-model--202606240/results.jsonl


### Download inference results

### Track Inference Results

We track inference results in a CSV file for evaluation of different models later.
1. We need to maintain a history of all inference runs
2. We want to associate results with specific models
3. We need to easily locate model outputs later
4. CSV format enables easy tracking

In [ ]:
!gsutil cat gs://electric-bill-dataset-gcs/inference_results/infer-gs:--electric-bill-dataset-gcs-output-model--202606240/results.jsonl > results.jsonl

In [ ]:
with open("results.jsonl", "r", encoding="utf-8") as f:
    print(f.read(1000))

{"response": "{\"account_holder_name\": \"Regina Williamsen\", \"account_number\": \"9215687\", \"amount_due\": \"50.73\", \"average_daily_usage_kwh\": \"1.216\", \"bill_due_date\": \"2025-03-17\", \"bill_issue_date\": \"2025-02-25\", \"billing_period_end\": \"2025-02-17\", \"billing_period_start\": \"2025-01-19\", \"document_type\": \"AUS_ELECTRICITY_BILL\", \"nmi\": \"61939939867\", \"provider_abn\": \"86 601 199 151\", \"provider_name\": \"Sumo Power Pty Ltd\", \"service_address_postcode\": \"3064\", \"service_address_state\": \"VIC\", \"service_address_street\": \"Apt. 059 68 Dylan Dip\", \"service_address_suburb\": \"Roxburgh Park\", \"supply_charge_days\": \"29\", \"tax_invoice_number\": \"5574906\", \"total_electricity_kwh\": \"35.231\", \"total_excl_gst\": \"46.12\", \"total_gst\": \"4.61\", \"total_incl_gst\": \"50.73\", \"usage_discount_amount\": null, \"usage_discount_percent\": null}", "labels": "{\"account_holder_name\": \"Regina Williamson\", \"account_number\": \"9215687

In [ ]:
!gsutil cp "gs://electric-bill-dataset-gcs/inference_results/infer-gs:--electric-bill-dataset-gcs-output-model--202606240/results.jsonl" .

Copying gs://electric-bill-dataset-gcs/inference_results/infer-gs:--electric-bill-dataset-gcs-output-model--202606240/results.jsonl...
/ [0 files][    0.0 B/579.0 KiB]                                                
/ [1 files][579.0 KiB/579.0 KiB]                                                

Operation completed over 1 objects/579.0 KiB.                                    


## Next step
* Continue with the [05_evaluate_model.ipynb](./05_evaluate_model.ipynb) notebook to evaluate the models performance and compare it to other models. 